# Feature comparison — Aligned vs Naive (per-feature)

두 CSV (`features_zones.csv`, `features_zones_naive.csv`) 를 로드해서 **feature 하나하나** timeline + CV bar 로 시각화.

**Feature 카테고리** (총 ~120):
- Density (11 zones × 3 metric = 33 핵심 + area/ratio = 88 total)
- Skeleton whole (3 network × 3 metric = 9)
- Caliber whole (3 network × 6 stat = 18)
- CRAE / CRVE / AVR + n_seg = 5

**시각화 스타일** (모든 feature 공통):
- Timeline: L aligned (파랑 실선), R aligned (주황 실선), L naive (회색 점선), R naive (회색 점선)
- CV bar: naive 앞 gray, aligned 뒤 파랑, reduction% 표시


## 0. Imports + CSV 로드


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PKG_ROOT = Path("..").resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from features.zones import ETDRS_9_NAMES, DISC_ZONE_NAMES

SQUARE_SIZE = 512
src_name = "20230118_HCS_Fundus_image"
features_out = Path(f"g:/RAG/project_1/output/features_{SQUARE_SIZE}") / src_name
compare_out  = Path(f"g:/RAG/project_1/output/features_compare_{SQUARE_SIZE}") / src_name
compare_out.mkdir(parents=True, exist_ok=True)

aligned_csv = features_out / "features_zones.csv"
naive_csv   = features_out / "features_zones_naive.csv"
features_aligned = pd.read_csv(aligned_csv, parse_dates=["date"])
features_naive   = pd.read_csv(naive_csv,   parse_dates=["date"])
print(f"aligned: {len(features_aligned)} rows × {features_aligned.shape[1]} cols")
print(f"naive:   {len(features_naive)} rows × {features_naive.shape[1]} cols")

patient_ids = sorted(features_aligned["patient"].unique())
zone_names  = ETDRS_9_NAMES + DISC_ZONE_NAMES
print(f"patients: {patient_ids}\nzones: {zone_names}")


## 1. Feature 목록 자동 추출 + helper 함수


In [ ]:
meta_cols = {"id","patient","eye","date","shot","disc_diameter_px","mm_per_px"}
all_feature_cols = [c for c in features_aligned.columns if c not in meta_cols]

def categorize(col):
    # Tier 1+2 신규 카테고리
    if col.startswith("FAZ_"):
        return "faz"
    if col.startswith("crossings_"):
        return "crossings"
    if "tortuosity" in col:
        return "tortuosity"
    if "bifurc_" in col:
        return "bifurcation"
    if "width_cov" in col:
        return "width_variability"
    if "junc_X" in col or col.endswith("_junc_n"):
        return "junction_exp"
    # 기존
    if col in {"CRAE_um","CRVE_um","AVR","n_artery_segments","n_vein_segments"}:
        return "knudtson"
    if col.startswith("sholl_"):
        return "sholl"
    if col.startswith("whole_") and "fractal_dim" in col:
        return "fractal"
    if col.startswith("whole_") and any(k in col for k in [
        "mean_R","mean_G","mean_B","mean_intensity","R_G_ratio","R_B_ratio",
        "whiteness","AV_intensity_ratio","AV_whiteness_ratio","AV_RG_ratio"
    ]):
        return "color"
    if col.startswith("whole_") and "caliber" in col:
        return "caliber_whole"
    if col.startswith("whole_") and ("skel_length" in col or "n_branches" in col or "n_endpoints" in col):
        return "skeleton_whole"
    if col.startswith("whole_") and ("n_segments" in col or "seg_len" in col):
        return "segment_stats"
    if col.startswith("whole_"):
        return "density_whole"
    return "density_zone"

from collections import defaultdict
cat_map = defaultdict(list)
for c in all_feature_cols:
    cat_map[categorize(c)].append(c)
for k, v in sorted(cat_map.items()):
    print(f"  {k}: {len(v)} cols")


In [ ]:
def cv(series):
    m = series.mean()
    return 100 * series.std() / m if m else np.nan

def plot_feature_timeline(feature_col, patient, ax):
    """단일 feature timeline: 4 line (L/R × aligned/naive)."""
    for eye, style_a, style_n in [("L","-o","--o"), ("R","-s","--s")]:
        color_a = {"L": "tab:blue", "R": "tab:orange"}[eye]
        a = features_aligned[(features_aligned["patient"]==patient) & (features_aligned["eye"]==eye)]
        n = features_naive  [(features_naive  ["patient"]==patient) & (features_naive  ["eye"]==eye)]
        if len(a) and feature_col in a: ax.plot(a["date"], a[feature_col], style_a,
                                                 label=f"{eye} aligned",
                                                 markersize=3, linewidth=1.2, color=color_a)
        if len(n) and feature_col in n: ax.plot(n["date"], n[feature_col], style_n,
                                                 label=f"{eye} naive",
                                                 markersize=2.5, linewidth=0.8, alpha=0.6, color="gray")
    ax.set_title(feature_col, fontsize=8)
    ax.grid(alpha=0.3); ax.legend(fontsize=6, ncol=2)
    ax.tick_params(axis="x", rotation=45, labelsize=6)

def compute_cv_row(feature_col):
    """모든 (patient, eye) 평균 CV — naive vs aligned."""
    cvs_a, cvs_n = [], []
    for pid in patient_ids:
        for eye in ["L", "R"]:
            a = features_aligned[(features_aligned["patient"]==pid) & (features_aligned["eye"]==eye)]
            n = features_naive  [(features_naive  ["patient"]==pid) & (features_naive  ["eye"]==eye)]
            if len(a) and feature_col in a: cvs_a.append(cv(a[feature_col]))
            if len(n) and feature_col in n: cvs_n.append(cv(n[feature_col]))
    return {
        "feature": feature_col,
        "CV_naive":   np.nanmean(cvs_n) if cvs_n else np.nan,
        "CV_aligned": np.nanmean(cvs_a) if cvs_a else np.nan,
    }


## 2. Density — feature 별 timeline


In [ ]:
# Density 는 vessel/artery/vein density 3개만 subset (나머지 area/ratio/frac 은 CV 표에는 다 있음)
density_core_metrics = ["vessel_density", "artery_density", "vein_density"]
density_core_cols = [f"{z}_{m}" for z in zone_names for m in density_core_metrics]
density_core_cols = [c for c in density_core_cols if c in all_feature_cols]

# 환자별: 3 rows (vessel/artery/vein) × 11 cols (zones)
for pid in patient_ids:
    fig, axes = plt.subplots(len(density_core_metrics), len(zone_names),
                             figsize=(2.6*len(zone_names), 2.6*len(density_core_metrics)),
                             squeeze=False, sharex=True)
    for r, mb in enumerate(density_core_metrics):
        for c, zn in enumerate(zone_names):
            plot_feature_timeline(f"{zn}_{mb}", pid, axes[r, c])
    fig.suptitle(f"Patient {pid} — Density (vessel/artery/vein) × 11 zones", fontsize=13)
    plt.tight_layout(); plt.show()


## 3. Skeleton whole — feature 별 timeline


In [ ]:
for pid in patient_ids:
    fig, axes = plt.subplots(3, 3, figsize=(12, 8), squeeze=False, sharex=True)
    networks    = ["vessel", "artery", "vein"]
    skel_metrics = ["skel_length_px", "n_branches", "n_endpoints"]
    for r, net in enumerate(networks):
        for c, sm in enumerate(skel_metrics):
            plot_feature_timeline(f"whole_{net}_{sm}", pid, axes[r, c])
    fig.suptitle(f"Patient {pid} — Skeleton whole-image (3 net × 3 metric)", fontsize=13)
    plt.tight_layout(); plt.show()


## 4. Caliber whole — feature 별 timeline


In [ ]:
for pid in patient_ids:
    fig, axes = plt.subplots(3, 6, figsize=(20, 8), squeeze=False, sharex=True)
    networks    = ["vessel", "artery", "vein"]
    cal_stats   = ["caliber_mean_um","caliber_median_um","caliber_p75_um",
                   "caliber_p90_um","caliber_max_um","caliber_n_samples"]
    for r, net in enumerate(networks):
        for c, st in enumerate(cal_stats):
            plot_feature_timeline(f"whole_{net}_{st}", pid, axes[r, c])
    fig.suptitle(f"Patient {pid} — Caliber whole-image (3 net × 6 stat)", fontsize=13)
    plt.tight_layout(); plt.show()


## 5. CRAE / CRVE / AVR + segments — feature 별 timeline


In [ ]:
for pid in patient_ids:
    fig, axes = plt.subplots(1, 5, figsize=(20, 3.5), squeeze=False, sharex=True)
    for c, fcol in enumerate(["CRAE_um","CRVE_um","AVR","n_artery_segments","n_vein_segments"]):
        plot_feature_timeline(fcol, pid, axes[0, c])
        if fcol == "AVR":
            axes[0, c].axhline(0.67, color="k", ls=":", alpha=0.4)
    fig.suptitle(f"Patient {pid} — CRAE / CRVE / AVR + segments", fontsize=13)
    plt.tight_layout(); plt.show()


## 5. Fractal dim — feature 별 timeline


In [ ]:
# Fractal dim timeline — 3 network per patient, one plot each
fig, axes = plt.subplots(1, len(patient_ids), figsize=(5*len(patient_ids), 4), squeeze=False)
for col, pid in enumerate(patient_ids):
    ax = axes[0, col]
    for net, color in [("vessel","tab:green"),("artery","tab:red"),("vein","tab:blue")]:
        col_name = f"whole_{net}_fractal_dim"
        for eye, ls in [("L","-o"),("R","--o")]:
            for src, style_a in [("aligned", "-"), ("naive", "--")]:
                df = features_aligned if src == "aligned" else features_naive
                sub = df[(df["patient"]==pid) & (df["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                marker = "o" if eye == "L" else "s"
                color_use = color if src == "aligned" else "gray"
                lw = 1.4 if src == "aligned" else 0.8
                alpha = 1.0 if src == "aligned" else 0.5
                ax.plot(sub["date"], sub[col_name], style_a+marker,
                        color=color_use, label=f"{eye} {net} {src}",
                        markersize=3, linewidth=lw, alpha=alpha)
    ax.set_title(f"{pid} — Fractal dim")
    ax.grid(alpha=0.3); ax.legend(fontsize=6, ncol=3)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.suptitle("Fractal dim (whole): aligned vs naive", fontsize=13)
plt.tight_layout(); plt.show()


## 6. Sholl summary — feature 별 timeline


In [ ]:
# Sholl summary timeline — 2 center × 3 network × 4 stat
sholl_stats = ["sholl_max","sholl_r_max_px","sholl_auc","sholl_mean"]
for center in ["fovea","disc"]:
    for pid in patient_ids:
        fig, axes = plt.subplots(3, len(sholl_stats),
                                 figsize=(4*len(sholl_stats), 8), squeeze=False, sharex=True)
        for r, net in enumerate(["vessel","artery","vein"]):
            for c, st in enumerate(sholl_stats):
                ax = axes[r, c]
                col_name = f"sholl_{center}_{net}_{st}"
                plot_feature_timeline(col_name, pid, ax)
        fig.suptitle(f"Patient {pid} — Sholl {center} — 3 net × 4 stat", fontsize=12)
        plt.tight_layout(); plt.show()


## 6. **모든 feature CV bar** — naive vs aligned


In [ ]:
# 전체 feature CV 계산
cv_rows = [compute_cv_row(c) for c in all_feature_cols]
cv_all = pd.DataFrame(cv_rows)
cv_all["reduction_%"] = 100 * (cv_all["CV_naive"] - cv_all["CV_aligned"]) / cv_all["CV_naive"]
cv_all["category"]    = cv_all["feature"].map(categorize)
cv_all.to_csv(compare_out / "cv_all_features.csv", index=False)
print(f"CV table 저장: {len(cv_all)} features")
cv_all.groupby("category")[["CV_naive","CV_aligned","reduction_%"]].agg(["mean","median"]).round(1)


In [ ]:
def plot_cv_bars(sub_df, title, ax, y_lim_top=None):
    x = np.arange(len(sub_df))
    w = 0.4
    ax.bar(x - w/2, sub_df["CV_naive"],   w, label="naive",   color="tab:gray")
    ax.bar(x + w/2, sub_df["CV_aligned"], w, label="aligned", color="tab:blue")
    ax.set_xticks(x); ax.set_xticklabels(sub_df["feature"], rotation=90, fontsize=5)
    ax.set_ylabel("CV (%)"); ax.set_title(title)
    ax.grid(alpha=0.3, axis="y"); ax.legend()
    if y_lim_top: ax.set_ylim(0, y_lim_top)

categories_to_plot = [
    "density_whole", "density_zone",
    "skeleton_whole", "segment_stats",
    "caliber_whole", "knudtson",
    "fractal", "sholl", "color",
    "tortuosity", "faz", "crossings",
    "bifurcation", "width_variability", "junction_exp",
]
for cat in categories_to_plot:
    sub = cv_all[cv_all["category"] == cat].sort_values("feature").reset_index(drop=True)
    if len(sub) == 0: continue
    fig, ax = plt.subplots(figsize=(max(6, 0.18*len(sub)), 5))
    plot_cv_bars(sub, f"{cat} — CV per feature ({len(sub)} features)", ax)
    plt.tight_layout(); plt.show()


## 7. Top-K feature — alignment 로 가장 개선된 것


In [ ]:
top = cv_all.dropna(subset=["reduction_%"]).sort_values("reduction_%", ascending=False).head(30)
print("Top 30 alignment-reduced features (naive > aligned CV):")
display(top[["feature","category","CV_naive","CV_aligned","reduction_%"]].round(1))

bottom = cv_all.dropna(subset=["reduction_%"]).sort_values("reduction_%").head(20)
print("\nBottom 20 (alignment 이 오히려 CV 증가시킨 features):")
display(bottom[["feature","category","CV_naive","CV_aligned","reduction_%"]].round(1))


## 8. Trend similarity — Pearson r + slope agreement

**CV 는 fluctuation 크기만** 봄. Trend 유사도 (같은 방향으로 변하는지) 는 별개 문제.

**두 시나리오가 CV 로는 구분 안 됨**:
- (좋음) naive/aligned 둘 다 서서히 하락 → 같은 정보, aligned 만 노이즈 적음
- (나쁨) naive 는 flat, aligned 는 하락 → 서로 다른 정보!

**여기서 뽑는 지표** (per feature × per (patient, eye)):
- **Pearson r** — 시계열 상관 (형태 일치도)
- **Cosine similarity** on z-scored — 정규화된 shape 유사도
- **Slope (linear regression)** — 장기 trend 방향/속도
- **Slope direction match** — 같은 부호인지


In [ ]:
# Per-feature × per-(patient, eye) trend metrics 계산
def per_group_trend(feature_col):
    rows = []
    for pid in patient_ids:
        for eye in ["L", "R"]:
            a = features_aligned[(features_aligned["patient"]==pid) & (features_aligned["eye"]==eye)]
            n = features_naive  [(features_naive  ["patient"]==pid) & (features_naive  ["eye"]==eye)]
            if len(a) == 0 or len(n) == 0: continue
            if feature_col not in a.columns or feature_col not in n.columns: continue

            merged = a[["id","date",feature_col]].merge(
                n[["id",feature_col]], on="id", suffixes=("_a","_n")
            ).dropna().sort_values("date")
            if len(merged) < 3: continue

            a_ts = merged[f"{feature_col}_a"].values.astype(float)
            n_ts = merged[f"{feature_col}_n"].values.astype(float)

            # Pearson r
            if a_ts.std() > 0 and n_ts.std() > 0:
                r = float(np.corrcoef(a_ts, n_ts)[0, 1])
            else:
                r = np.nan

            # Cosine similarity on z-scored
            def zscore(x):
                s = x.std()
                return (x - x.mean()) / s if s > 0 else np.zeros_like(x)
            a_z, n_z = zscore(a_ts), zscore(n_ts)
            cos = float(np.dot(a_z, n_z) / len(a_z)) if len(a_z) > 0 else np.nan

            # Slope (days as x)
            t = (merged["date"] - merged["date"].min()).dt.days.values.astype(float)
            if len(t) >= 3 and t.std() > 0:
                slope_a = float(np.polyfit(t, a_ts, 1)[0])
                slope_n = float(np.polyfit(t, n_ts, 1)[0])
                dir_match = int(np.sign(slope_a) == np.sign(slope_n))
            else:
                slope_a = slope_n = np.nan
                dir_match = np.nan

            rows.append({
                "patient": pid, "eye": eye, "n_visits": len(merged),
                "pearson_r": r, "cosine_sim": cos,
                "slope_aligned": slope_a, "slope_naive": slope_n,
                "slope_direction_match": dir_match,
            })
    return rows

from tqdm import tqdm
trend_rows = []
for fcol in tqdm(all_feature_cols, desc="trend"):
    for r in per_group_trend(fcol):
        trend_rows.append({"feature": fcol, "category": categorize(fcol), **r})
trend_df = pd.DataFrame(trend_rows)
trend_df.to_csv(compare_out / "trend_per_group.csv", index=False)
print(f"per-group trend: {len(trend_df)} rows (features × groups)")
trend_df.head()


### Per-feature 집계 — 평균 Pearson r, slope 등


In [ ]:
trend_agg = (trend_df.groupby(["feature","category"], sort=False)
                    .agg(mean_r=("pearson_r","mean"),
                         median_r=("pearson_r","median"),
                         mean_cos=("cosine_sim","mean"),
                         mean_slope_a=("slope_aligned","mean"),
                         mean_slope_n=("slope_naive","mean"),
                         pct_dir_match=("slope_direction_match", lambda s: 100*np.nanmean(s)),
                         n_groups=("pearson_r","count"))
                    .reset_index())
trend_agg.to_csv(compare_out / "trend_per_feature.csv", index=False)
print(f"per-feature aggregate: {len(trend_agg)} features")
trend_agg.head()


### 카테고리별 Pearson r 분포 (boxplot)

**해석**:
- **r ≈ 1** → naive 와 aligned 가 같은 정보를 담고 있음. Alignment 는 노이즈만 줄이는 효과
- **r ≈ 0** → 두 방법이 완전히 다른 신호. Alignment 가 진짜 새 정보 제공
- **r < 0** → 반대 방향 → 심각 (alignment 에러거나 mask/segment 오작동)


In [ ]:
cats_ordered = ["density_whole","density_zone","skeleton_whole","segment_stats",
                "caliber_whole","knudtson","fractal","sholl","color",
                "tortuosity","faz","crossings","bifurcation","width_variability","junction_exp"]
cat_colors  = ["#8bd0ff","#3e88c9","#a8ff9c","#5cbf4f","#ffb08b","#cf5c1e","#dbb2ff","#8b3cc7","#ff69b4","#96ceb4","#ffe066","#e07b7b","#66ccff","#c47ac0","#8899a6"]
color_map = dict(zip(cats_ordered, cat_colors))

fig, ax = plt.subplots(figsize=(12, 6))
data = [trend_df[trend_df["category"]==c]["pearson_r"].dropna().values for c in cats_ordered]
bp = ax.boxplot(data, labels=cats_ordered, showmeans=True, patch_artist=True)
for patch, c in zip(bp["boxes"], cat_colors):
    patch.set_facecolor(c)
ax.axhline(0,   color="k",      lw=0.5)
ax.axhline(0.5, color="orange", ls="--", alpha=0.5, label="r=0.5")
ax.axhline(0.9, color="green",  ls="--", alpha=0.5, label="r=0.9")
ax.set_ylabel("Pearson r (aligned vs naive)")
ax.set_title("Trend agreement by category — 낮을수록 alignment 가 다른 정보 제공")
ax.grid(alpha=0.3, axis="y"); ax.legend()
ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

print("\nPer-category mean:")
print(trend_agg.groupby("category")[["mean_r","median_r","pct_dir_match"]].mean().round(3))


### Slope agreement scatter

대각선 (y=x) 위에 있으면 같은 방향/속도 → alignment 무관.
벗어날수록 alignment 로 slope 자체가 바뀜.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
for cat in cats_ordered:
    sub = trend_df[trend_df["category"]==cat].dropna(subset=["slope_naive","slope_aligned"])
    if len(sub) == 0: continue
    ax.scatter(sub["slope_naive"], sub["slope_aligned"], s=6, alpha=0.6,
               color=color_map[cat], label=cat)

lim_all = np.concatenate([
    trend_df["slope_naive"].dropna().values, trend_df["slope_aligned"].dropna().values
])
lim_p = np.percentile(np.abs(lim_all[np.isfinite(lim_all)]), 99) if len(lim_all) else 1
ax.plot([-lim_p, lim_p], [-lim_p, lim_p], "k--", lw=0.8, alpha=0.6, label="y=x")
ax.axhline(0, color="k", lw=0.3); ax.axvline(0, color="k", lw=0.3)
ax.set_xlim(-lim_p, lim_p); ax.set_ylim(-lim_p, lim_p)
ax.set_xlabel("slope naive (per day)"); ax.set_ylabel("slope aligned (per day)")
ax.set_title("Slope: aligned vs naive")
ax.grid(alpha=0.3); ax.legend(loc="upper left", ncol=2, fontsize=8)
plt.tight_layout(); plt.show()


### 카테고리별 slope direction 일치율 (%)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
per_cat = (trend_df.groupby("category")["slope_direction_match"]
                   .apply(lambda s: 100*np.nanmean(s))
                   .reindex(cats_ordered))
ax.bar(range(len(per_cat)), per_cat.values,
       color=[color_map[c] for c in per_cat.index])
ax.set_xticks(range(len(per_cat))); ax.set_xticklabels(per_cat.index, rotation=15)
ax.set_ylabel("% direction match (naive vs aligned)")
ax.axhline(50, color="red", ls="--", alpha=0.5, label="random 50%")
ax.set_ylim(0, 105)
ax.set_title("Trend direction agreement — 낮으면 alignment 가 방향 자체를 바꿈")
ax.grid(alpha=0.3, axis="y"); ax.legend()
for i, v in enumerate(per_cat.values):
    if np.isfinite(v):
        ax.text(i, v+1, f"{v:.0f}%", ha="center", fontsize=9)
plt.tight_layout(); plt.show()


### Top-K disagreeing / agreeing features

- **낮은 r (< 0.5)** = alignment 가 다른 정보 제공 (진짜 이득)
- **높은 r (> 0.9)** = 두 방법이 같은 신호. Alignment 는 노이즈 감소만


In [ ]:
disag = trend_agg.dropna(subset=["mean_r"]).sort_values("mean_r").head(25)
print("가장 disagreeing (r 낮음):")
display(disag[["feature","category","mean_r","median_r","pct_dir_match","n_groups"]].round(3))

print("\n가장 agreeing (r 높음):")
agree = trend_agg.dropna(subset=["mean_r"]).sort_values("mean_r", ascending=False).head(20)
display(agree[["feature","category","mean_r","median_r","pct_dir_match","n_groups"]].round(3))


### Alignment 효과 종합 — CV reduction + trend disagreement

**진짜로 alignment 가 값 있는 feature**:
- CV_naive > CV_aligned (fluctuation 감소) AND
- Pearson r 이 낮음 (다른 정보)

두 조건 다 만족하면 alignment 없이는 신호 못 뽑았다는 강력한 증거


In [ ]:
merged_scores = cv_all.merge(
    trend_agg[["feature","mean_r","pct_dir_match"]], on="feature", how="left")
merged_scores["alignment_value"] = (
    merged_scores["reduction_%"].fillna(0) * (1 - merged_scores["mean_r"].fillna(1))
)
merged_scores.to_csv(compare_out / "alignment_value_score.csv", index=False)

top_val = merged_scores.dropna(subset=["alignment_value"]).sort_values(
    "alignment_value", ascending=False).head(30)
print("Alignment 로부터 가장 이득이 큰 features (CV 감소 크고 trend 다름):")
display(top_val[["feature","category","CV_naive","CV_aligned","reduction_%","mean_r","alignment_value"]].round(2))


## 10. Alignment 최종 판정 (Verdict) - feature 분류

CV reduction + Pearson r 조합. **Hurt 조건은 AND 로 강화** (두 지표 모두 나빠야).

| Verdict | 조건 | 해석 |
|---|---|---|
| ⭐ Benefit | reduction ≥ 20% AND mean_r < 0.7 | Alignment 실제 이득 |
| ➖ Similar | \|reduction\| < 10% AND mean_r > 0.9 | 두 방법 거의 동일 |
| 🔀 Info_different | mean_r < 0.3 AND \|reduction\| < 10 | 다른 신호 (좋은지 나쁜지 불명) |
| ⚠️ Hurt | reduction < -20% AND mean_r < 0.5 | 둘 다 나빠야 hurt |
| 🔵 Mixed | 나머지 | |
| 🔧 Metadata | `_n_samples` 계열 | Raw pixel count 제외 |

**중요**: 낮은 r 만으로는 hurt 로 판정 안 함 - aligned 가 다른 anatomical 신호를 잡는 것일 수 있음 (info_different 카테고리).


In [ ]:
def is_count_metadata(col):
    keys = ["_n_samples"]      # skeleton pixel 수 raw count 만 제외
    return any(k in col for k in keys)

verdict_df = cv_all.merge(trend_agg[["feature","mean_r","mean_cos","pct_dir_match"]],
                          on="feature", how="left").copy()
verdict_df["category"] = verdict_df["feature"].map(categorize)
verdict_df["is_metadata"] = verdict_df["feature"].map(is_count_metadata)

n_meta = int(verdict_df["is_metadata"].sum())
print(f"메타 지표 (verdict 제외): {n_meta} features")

# Hurt: 이제 AND 로 강화 (CV 도 나쁘고 r 도 낮아야)
def classify(row):
    if row["is_metadata"]:
        return "metadata"
    red = row.get("reduction_%", float("nan"))
    r   = row.get("mean_r", float("nan"))
    if pd.isna(red) and pd.isna(r):
        return "unknown"
    if pd.notna(red) and red >= 20 and (pd.isna(r) or r < 0.7):
        return "benefit"
    if pd.notna(red) and abs(red) < 10 and pd.notna(r) and r > 0.9:
        return "similar"
    if pd.notna(red) and red < -20 and pd.notna(r) and r < 0.5:
        return "hurt"
    if pd.notna(r) and r < 0.3 and (pd.isna(red) or red >= -10):
        return "info_different"
    return "mixed"

verdict_df["verdict"] = verdict_df.apply(classify, axis=1)
verdict_df.to_csv(compare_out / "alignment_verdict.csv", index=False)

counts = verdict_df["verdict"].value_counts().reindex(
    ["benefit","similar","info_different","mixed","hurt","unknown","metadata"], fill_value=0)
print("=" * 55)
print("Alignment verdict 요약 (hurt 조건 AND 로 강화)")
print("=" * 55)
for v, n in counts.items():
    emoji = {"benefit":"⭐","similar":"➖","info_different":"🔀","mixed":"🔵",
             "hurt":"⚠️","unknown":"?","metadata":"🔧"}[v]
    pct = 100 * n / len(verdict_df) if len(verdict_df) else 0
    print(f"  {emoji} {v:16s}: {n:>4} features ({pct:.1f}%)")

bio = verdict_df[~verdict_df["is_metadata"]]
pivot = bio.pivot_table(index="category", columns="verdict",
                        values="feature", aggfunc="count", fill_value=0)
pivot = pivot.reindex(columns=["benefit","similar","info_different","mixed","hurt","unknown"],
                     fill_value=0)
print("\n카테고리별 verdict 분포 (메타 제외):")
print(pivot)


In [ ]:
pivot

### 10a. ⭐ Benefit features — alignment 실제로 이득


In [ ]:
benefit_df = verdict_df[(verdict_df["verdict"] == "benefit") & (~verdict_df["is_metadata"])].sort_values("reduction_%", ascending=False)
print(f"⭐ Benefit features: {len(benefit_df)}")
if len(benefit_df) > 0:
    print(f"\n카테고리 분포:")
    print(benefit_df["category"].value_counts())
    print(f"\n상위 20:")
    display(benefit_df[["feature","category","CV_naive","CV_aligned","reduction_%","mean_r"]].head(20).round(2))

# Benefit features CV bar chart
if len(benefit_df) > 0:
    fig, ax = plt.subplots(figsize=(max(8, 0.2*len(benefit_df)), 6))
    x = np.arange(len(benefit_df))
    w = 0.4
    ax.bar(x - w/2, benefit_df["CV_naive"],   w, label="naive",   color="tab:gray")
    ax.bar(x + w/2, benefit_df["CV_aligned"], w, label="aligned", color="tab:blue")
    ax.set_xticks(x); ax.set_xticklabels(benefit_df["feature"], rotation=90, fontsize=5)
    ax.set_ylabel("CV (%)"); ax.set_title(f"⭐ Benefit features ({len(benefit_df)}) — CV 비교")
    ax.grid(alpha=0.3, axis="y"); ax.legend()
    plt.tight_layout(); plt.show()

    # Timeline for top-6 benefit
    top6 = benefit_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} — Top-6 benefit features", fontsize=12)
        plt.tight_layout(); plt.show()


### 10b. ➖ Similar features — 두 방법 거의 동일


In [ ]:
similar_df = verdict_df[(verdict_df["verdict"] == "similar") & (~verdict_df["is_metadata"])].sort_values("mean_r", ascending=False)
print(f"➖ Similar features: {len(similar_df)}")
if len(similar_df) > 0:
    print(f"\n카테고리 분포:")
    print(similar_df["category"].value_counts())
    print(f"\n상위 20 (r 가장 높은 것):")
    display(similar_df[["feature","category","CV_naive","CV_aligned","reduction_%","mean_r"]].head(20).round(2))

if len(similar_df) > 0:
    fig, ax = plt.subplots(figsize=(max(8, 0.2*len(similar_df)), 6))
    x = np.arange(len(similar_df))
    w = 0.4
    ax.bar(x - w/2, similar_df["CV_naive"],   w, label="naive",   color="tab:gray")
    ax.bar(x + w/2, similar_df["CV_aligned"], w, label="aligned", color="tab:blue")
    ax.set_xticks(x); ax.set_xticklabels(similar_df["feature"], rotation=90, fontsize=5)
    ax.set_ylabel("CV (%)"); ax.set_title(f"➖ Similar features ({len(similar_df)}) — CV 비교 (거의 겹침)")
    ax.grid(alpha=0.3, axis="y"); ax.legend()
    plt.tight_layout(); plt.show()

    # Timeline for a few — 두 line 이 거의 겹쳐야 정상
    top6 = similar_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} — Top-6 similar features (aligned == naive)", fontsize=12)
        plt.tight_layout(); plt.show()


### 10c. ⚠️ Hurt features — alignment 가 오히려 안 좋아짐 (드묾)


In [ ]:
hurt_df = verdict_df[(verdict_df["verdict"] == "hurt") & (~verdict_df["is_metadata"])].sort_values("reduction_%")
print(f"⚠️ Hurt features: {len(hurt_df)} (조건: reduction < -20% AND r < 0.5)")
if len(hurt_df) > 0:
    print(f"\n카테고리 분포:")
    print(hurt_df["category"].value_counts())
    display(hurt_df[["feature","category","CV_naive","CV_aligned","reduction_%","mean_r"]].head(30).round(2))

    fig, ax = plt.subplots(figsize=(max(8, 0.2*len(hurt_df)), 5))
    x = np.arange(len(hurt_df))
    w = 0.4
    ax.bar(x - w/2, hurt_df["CV_naive"],   w, label="naive",   color="tab:gray")
    ax.bar(x + w/2, hurt_df["CV_aligned"], w, label="aligned", color="tab:red")
    ax.set_xticks(x); ax.set_xticklabels(hurt_df["feature"], rotation=90, fontsize=6)
    ax.set_ylabel("CV (%)")
    ax.set_title(f"⚠️ Hurt features ({len(hurt_df)}) — aligned worse in BOTH CV & r")
    ax.grid(alpha=0.3, axis="y"); ax.legend()
    plt.tight_layout(); plt.show()

    top6 = hurt_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} — Top-6 hurt features (worst reduction)", fontsize=12)
        plt.tight_layout(); plt.show()
else:
    print("(hurt features 없음 - 좋은 신호, alignment 파이프라인 안정적)")


### 10d. 🔀 Info_different features - 다른 신호, 어느게 옳은지 불명

`mean_r < 0.3` AND `|reduction| < 10` - CV 는 비슷하지만 시계열 형태가 크게 다름.

- Naive 시계열 = 매 visit segmentation 노이즈 따라 움직임
- Aligned 시계열 = fixed anatomical 좌표에서 값이 안정 (다른 신호 캡처)
- **어느게 진짜 signal 인지는 ground truth 없이 판단 불가** - 별도 분석 필요


In [ ]:
info_df = verdict_df[(verdict_df["verdict"] == "info_different") & (~verdict_df["is_metadata"])].sort_values("mean_r")
print(f"🔀 Info_different features: {len(info_df)}")
if len(info_df) > 0:
    print(f"\n카테고리 분포:")
    print(info_df["category"].value_counts())
    display(info_df[["feature","category","CV_naive","CV_aligned","reduction_%","mean_r"]].head(20).round(2))

    top6 = info_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} - Top-6 info_different (다른 신호)", fontsize=12)
        plt.tight_layout(); plt.show()


### 10d. Verdict pie chart + 카테고리 별 stacked bar


In [ ]:
# 전체 pie
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

verdict_order = ["benefit","similar","info_different","mixed","hurt","unknown","metadata"]
verdict_colors = ["#4caf50","#9e9e9e","#ba68c8","#03a9f4","#f44336","#cccccc","#e0c46a"]
counts_val = counts.reindex(verdict_order, fill_value=0)
mask = counts_val > 0

axes[0].pie(counts_val[mask], labels=[f"{v} ({n})" for v, n in counts_val[mask].items()],
            colors=[c for c, m in zip(verdict_colors, mask) if m],
            autopct="%1.0f%%", startangle=90)
axes[0].set_title(f"전체 alignment verdict ({len(verdict_df)} features)", fontsize=12)

# 카테고리별 stacked bar
pivot.plot(kind="bar", stacked=True, ax=axes[1],
           color=verdict_colors[:len(pivot.columns)])
axes[1].set_ylabel("# features"); axes[1].set_title("카테고리별 verdict 분포")
axes[1].grid(alpha=0.3, axis="y"); axes[1].legend(title="verdict")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


## 11. Trend revelation — alignment 이 age-related 신호 드러내는가

**핵심 관찰**: CV reduction 크기 자체보다 **"aligned 시계열이 age 와 monotonic trend 를 갖는가"** 가 진짜 신호.

**Naive vs Aligned 세 시나리오**:

| Aligned time-r | Naive time-r | Divergence with age | 해석 |
|---|---|---|---|
| 높음 | 낮음 | 큼 | ⭐ Alignment 가 biological trend 드러냄 (naive 는 노이즈로 묻힘) |
| 높음 | 높음 | 작음 | 두 방법 다 trend 잘 봄 (alignment 필요 없음) |
| 낮음 | 낮음 | 작음 | Feature 자체가 stationary (trend 없음) |
| 낮음 | 높음 | 큼 | ⚠️ Aligned 가 신호를 오히려 지움 (드묾) |

**새 지표**:
- `time_r_aligned` = |Pearson r (time_days, aligned_series)|
- `time_r_naive` = |Pearson r (time_days, naive_series)|
- `time_r_gain` = time_r_aligned - time_r_naive (alignment 이 준 추가 trend)
- `divergence_r` = |Pearson r (time_days, |aligned - naive|)|


In [ ]:
from tqdm import tqdm as _tqdm

def per_group_time_trend(feature_col):
    rows = []
    for pid in patient_ids:
        for eye in ["L", "R"]:
            a = features_aligned[(features_aligned["patient"]==pid) & (features_aligned["eye"]==eye)]
            n = features_naive  [(features_naive  ["patient"]==pid) & (features_naive  ["eye"]==eye)]
            if len(a) == 0 or len(n) == 0: continue
            if feature_col not in a.columns or feature_col not in n.columns: continue

            merged = a[["id","date",feature_col]].merge(
                n[["id",feature_col]], on="id", suffixes=("_a","_n")
            ).dropna().sort_values("date")
            if len(merged) < 5: continue

            t = (merged["date"] - merged["date"].min()).dt.days.values.astype(float)
            if t.std() == 0: continue
            a_ts = merged[f"{feature_col}_a"].values.astype(float)
            n_ts = merged[f"{feature_col}_n"].values.astype(float)
            diff_ts = np.abs(a_ts - n_ts)

            def _r(x, y):
                if x.std() > 0 and y.std() > 0:
                    return float(np.corrcoef(x, y)[0, 1])
                return np.nan

            rows.append({
                "patient": pid, "eye": eye, "n": len(merged),
                "time_r_aligned": _r(t, a_ts),
                "time_r_naive":   _r(t, n_ts),
                "divergence_r":   _r(t, diff_ts),
            })
    return rows

trend_rev_rows = []
for fcol in _tqdm(all_feature_cols, desc="trend-revelation"):
    for r in per_group_time_trend(fcol):
        trend_rev_rows.append({"feature": fcol, "category": categorize(fcol), **r})
trend_rev_df = pd.DataFrame(trend_rev_rows)

# 절댓값으로 집계
for c in ["time_r_aligned","time_r_naive","divergence_r"]:
    trend_rev_df[f"abs_{c}"] = trend_rev_df[c].abs()

trend_rev_agg = (trend_rev_df.groupby(["feature","category"], sort=False)
                 .agg(mean_time_r_aligned=("abs_time_r_aligned","mean"),
                      mean_time_r_naive  =("abs_time_r_naive",  "mean"),
                      mean_divergence_r  =("abs_divergence_r",  "mean"),
                      n_groups           =("time_r_aligned",    "count"))
                 .reset_index())
trend_rev_agg["time_r_gain"] = trend_rev_agg["mean_time_r_aligned"] - trend_rev_agg["mean_time_r_naive"]

trend_rev_df.to_csv(compare_out / "trend_revelation_per_group.csv", index=False)
trend_rev_agg.to_csv(compare_out / "trend_revelation_per_feature.csv", index=False)
print(f"trend_rev_df: {len(trend_rev_df)} rows (feature × group)")
print(f"trend_rev_agg: {len(trend_rev_agg)} features")
trend_rev_agg.head()


### 11a. Trend revelation 분류

- **⭐ Revealed** = aligned time_r 강함 (≥0.5) AND naive time_r 약함 (<0.3) → alignment 로 trend 드러남
- **➖ Both trend** = 둘 다 강함 (≥0.5) → naive 도 봄, alignment 필요 없음
- **📉 Stationary** = 둘 다 약함 (<0.3) → 원래 trend 없는 feature
- **⚠️ Erased** = naive 강함, aligned 약함 → alignment 가 오히려 지움 (드묾)
- **🔵 Mixed** = 나머지


In [ ]:
def classify_trend(row):
    ra = row["mean_time_r_aligned"]
    rn = row["mean_time_r_naive"]
    if pd.isna(ra) or pd.isna(rn): return "unknown"
    if ra >= 0.5 and rn <  0.3: return "revealed"
    if ra >= 0.5 and rn >= 0.5: return "both_trend"
    if ra <  0.3 and rn <  0.3: return "stationary"
    if ra <  0.3 and rn >= 0.5: return "erased"
    return "mixed"

trend_rev_agg["trend_verdict"] = trend_rev_agg.apply(classify_trend, axis=1)
counts = trend_rev_agg["trend_verdict"].value_counts().reindex(
    ["revealed","both_trend","stationary","erased","mixed","unknown"], fill_value=0)
print("=" * 55)
print("Trend revelation 판정")
print("=" * 55)
emoji = {"revealed":"⭐","both_trend":"➖","stationary":"📉","erased":"⚠️","mixed":"🔵","unknown":"?"}
for v, n in counts.items():
    print(f"  {emoji[v]} {v:12s}: {n:>4} features")

# 카테고리별 분포
pivot = trend_rev_agg.pivot_table(index="category", columns="trend_verdict",
                                   values="feature", aggfunc="count", fill_value=0)
pivot = pivot.reindex(columns=["revealed","both_trend","stationary","erased","mixed"], fill_value=0)
print("\n카테고리별:")
print(pivot)


### 11b. ⭐ Revealed features — alignment 로 age trend 드러남


In [ ]:
revealed_df = trend_rev_agg[trend_rev_agg["trend_verdict"] == "revealed"].sort_values(
    "time_r_gain", ascending=False)
print(f"⭐ Revealed features: {len(revealed_df)}")
if len(revealed_df) > 0:
    print("\n카테고리 분포:")
    print(revealed_df["category"].value_counts())
    print("\n상위 20 (time_r_gain 큰 순):")
    display(revealed_df.head(20)[["feature","category","mean_time_r_aligned",
                                    "mean_time_r_naive","time_r_gain","mean_divergence_r"]].round(3))

    # Timeline for top-6 revealed features
    top6 = revealed_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} — Top-6 REVEALED features (aligned=clean trend, naive=noise)",
                     fontsize=12)
        plt.tight_layout(); plt.show()
else:
    print("(alignment 로 새로 드러난 age-trend feature 없음)")


### 11c. ⚠️ Erased features — alignment 가 오히려 trend 지움 (드묾)


In [ ]:
erased_df = trend_rev_agg[trend_rev_agg["trend_verdict"] == "erased"].sort_values(
    "time_r_gain")
print(f"⚠️ Erased features: {len(erased_df)}")
if len(erased_df) > 0:
    print("\n카테고리 분포:")
    print(erased_df["category"].value_counts())
    display(erased_df.head(20)[["feature","category","mean_time_r_aligned",
                                  "mean_time_r_naive","time_r_gain"]].round(3))

    top6 = erased_df.head(6)["feature"].tolist()
    for pid in patient_ids:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
        for ax, fcol in zip(axes.flat, top6):
            plot_feature_timeline(fcol, pid, ax)
        fig.suptitle(f"Patient {pid} — Top-6 ERASED (aligned 가 trend 를 감춤?)", fontsize=12)
        plt.tight_layout(); plt.show()
else:
    print("(erased 없음 - 좋은 신호)")


### 11d. Time-r scatter — 전체 feature 을 (naive time-r, aligned time-r) 좌표에 배치


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
verdict_colors_tr = {
    "revealed": "#4caf50", "both_trend": "#9e9e9e",
    "stationary": "#03a9f4", "erased": "#f44336", "mixed": "#ba68c8",
}
for verdict, color in verdict_colors_tr.items():
    sub = trend_rev_agg[trend_rev_agg["trend_verdict"] == verdict]
    if len(sub) == 0: continue
    ax.scatter(sub["mean_time_r_naive"], sub["mean_time_r_aligned"],
               s=15, alpha=0.7, color=color, label=f"{verdict} ({len(sub)})")

# 판정 경계선
ax.axhline(0.5, color="green",  ls="--", alpha=0.4)  # aligned strong trend threshold
ax.axhline(0.3, color="orange", ls="--", alpha=0.4)  # aligned weak
ax.axvline(0.5, color="green",  ls="--", alpha=0.4)
ax.axvline(0.3, color="orange", ls="--", alpha=0.4)
ax.plot([0,1], [0,1], "k-", alpha=0.3, label="y=x")

ax.set_xlabel("time_r naive  (naive 시계열의 age 상관)")
ax.set_ylabel("time_r aligned (aligned 시계열의 age 상관)")
ax.set_title("Feature 별 age-trend 상관 — aligned 가 위로 갈수록 alignment 이득 큼")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()

# Top time_r_gain (아래 오른쪽 → 위 왼쪽 이동)
print("\nTop 15 time_r_gain (alignment 로 trend 가장 크게 드러남):")
display(trend_rev_agg.dropna(subset=["time_r_gain"]).sort_values(
    "time_r_gain", ascending=False).head(15)[
    ["feature","category","mean_time_r_naive","mean_time_r_aligned","time_r_gain"]
].round(3))


## 9. 요약

**CV 기반 저장물**:
- `compare_out/cv_all_features.csv` — 모든 feature CV_naive/CV_aligned/reduction_%

**Trend similarity 저장물**:
- `compare_out/trend_per_group.csv` — per feature × per (patient, eye) Pearson r, cosine, slope, dir match
- `compare_out/trend_per_feature.csv` — per feature 평균
- `compare_out/alignment_value_score.csv` — CV reduction × (1-r) 조합 지표

**해석 매트릭스**:

| CV reduction | Pearson r | 결론 |
|---|---|---|
| 높음 | 낮음 | ⭐ **Alignment 진짜 이득** (다른 정보 + 노이즈 감소) |
| 높음 | 높음 | Naive 도 같은 신호. Alignment 는 노이즈만 줄임 |
| 낮음 | 높음 | Alignment 효과 미미 — 이 feature 는 원래 robust |
| 낮음 | 낮음 | Alignment 가 오히려 다른 (틀린?) 방향으로 왜곡 |

**Top-K alignment_value** 로 진짜로 alignment 파이프라인의 가치가 있는 feature 를 정량적으로 선정 가능.
